# Telko — Ollama sur Google Colab (GPU T4)

Ce notebook installe Ollama sur le GPU Colab et l'expose via ngrok.
Une fois lancé, copie l'URL ngrok dans ton `.env` local.

> **Prérequis :** Runtime → Changer le type d'exécution → GPU T4

In [ ]:
# Cellule 1 — Installer Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Cellule 2 — Démarrer Ollama en arrière-plan
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(3)
print("Ollama démarré")

In [ ]:
# Cellule 3 — Télécharger les modèles
!ollama pull mistral
!ollama pull nomic-embed-text

In [ ]:
# Cellule 4 — Installer et configurer ngrok
!pip install pyngrok
from pyngrok import ngrok

# Colle ton token ngrok ici (compte gratuit sur ngrok.com)
ngrok.set_auth_token("TON_TOKEN_NGROK_ICI")

# Expose Ollama sur internet
tunnel = ngrok.connect(11434, "http")
print(f"URL publique Ollama : {tunnel.public_url}")
print(f"Ajoute dans ton .env : OLLAMA_BASE_URL={tunnel.public_url}")

In [ ]:
# Cellule 5 — Test de vérification
import requests
response = requests.post(
    f"{tunnel.public_url}/api/chat",
    json={
        "model": "mistral",
        "messages": [{"role": "user", "content": "Dis bonjour"}],
        "stream": False
    }
)
print(response.json()["message"]["content"])

In [ ]:
# Cellule 6 — Keepalive (empêche Colab de s'endormir)
import time
print("Ollama actif. Garde cet onglet ouvert.")
while True:
    requests.get(f"{tunnel.public_url}/api/tags")
    time.sleep(30)